In [ ]:
#!/usr/bin/env python3
# encoding: utf-8
#
# Copyright (C) 2024 Max Planck Institute for Multidisclplinary Sciences
# Copyright (C) 2024 Bharti Arora <bharti.arora@mpinat.mpg.de>
# Copyright (C) 2024 Ajinkya Kulkarni <ajinkya.kulkarni@mpinat.mpg.de>
#
# This program is free software: you can redistribute it and/or modify
# it under the terms of the GNU Affero General Public License as
# published by the Free Software Foundation, either version 3 of the
# License, or (at your option) any later version.
#
# This program is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
# GNU Affero General Public License for more details.
#
# You should have received a copy of the GNU Affero General Public License
# along with this program. If not, see <https://www.gnu.org/licenses/>.

########################################################################################

In [ ]:
import sys
# Don't generate the __pycache__ folder locally
sys.dont_write_bytecode = True
# Print exception without the built-in python warning
sys.tracebacklimit = 0

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

from PIL import Image
Image.MAX_IMAGE_PIXELS = None

import os
import glob
from tqdm.auto import tqdm

from modules_2photon import *

In [ ]:
# ── Parameters ──────────────────────────────────────────────────────────────────
FilterKey     = 2    # Gaussian sigma for pre-filtering
LocalSigmaKey = 10   # Structure tensor integration scale

# Quiver plot: every SpacingKey-th pixel is drawn as an arrow
SpacingKey    = 20
# Arrow length scaling (larger value = shorter arrows)
ScaleKey      = 50

folder_path   = 'Data'

In [ ]:
all_tif_files = glob.glob(os.path.join(folder_path, '*.tif'))

exclude_prefixes = ['FilteredImage_', 'DensityImage_', 'CoheranceImage_',
                    'OrientationImage_', 'CircularVarianceImage_', 'Results_']

filenames = [f for f in all_tif_files
             if not any(os.path.basename(f).startswith(p) for p in exclude_prefixes)]
filenames = sorted(filenames)

print(filenames)

In [ ]:
for uploaded_file_path in tqdm(filenames, desc='Analyzing files', leave=True):

    raw_image = np.array(Image.open(uploaded_file_path))
    raw_image = 255 * ((raw_image - raw_image.min()) / (raw_image.max() - raw_image.min()))

    ThresholdValueKey = max(int(np.median(raw_image)), 2)

    filtered_image = make_filtered_image(raw_image, FilterKey)

    # Compute structure tensor, coherence, orientation, and eigenvectors
    image_gradient_x, image_gradient_y = make_image_gradients(filtered_image)

    Structure_Tensor, EigenValues, EigenVectors, Jxx, Jxy, Jyy = make_structure_tensor_2d(
        image_gradient_x, image_gradient_y, LocalSigmaKey)

    Image_Coherance = make_coherence(
        filtered_image, EigenValues, Structure_Tensor, ThresholdValueKey)

    Image_Orientation = make_orientation(
        filtered_image, Jxx, Jxy, Jyy, ThresholdValueKey)

    # vx, vy are the principal eigenvector components at each pixel
    vx, vy = make_vxvy(filtered_image, EigenVectors, ThresholdValueKey)

    # ── Plot: SHG image + coherence + orientation quiver overlay ──────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Panel 1: raw SHG image
    axes[0].imshow(raw_image, cmap='gray', vmin=0, vmax=255)
    axes[0].set_title('SHG Image', fontsize=13)
    axes[0].set_xticks([])
    axes[0].set_yticks([])

    # Panel 2: coherence heatmap
    im = axes[1].imshow(Image_Coherance, cmap='inferno', vmin=0, vmax=1)
    axes[1].set_title('Coherence', fontsize=13)
    axes[1].set_xticks([])
    axes[1].set_yticks([])
    divider = make_axes_locatable(axes[1])
    cax = divider.append_axes('right', size='5%', pad=0.1)
    fig.colorbar(im, cax=cax)

    # Panel 3: SHG image with orientation quiver overlay
    # Headless arrows = undirected local fibre orientation
    axes[2].imshow(raw_image, cmap='gray', vmin=0, vmax=255, alpha=0.7)

    xmesh, ymesh = np.meshgrid(
        np.arange(raw_image.shape[0]),
        np.arange(raw_image.shape[1]),
        indexing='ij'
    )

    axes[2].quiver(
        ymesh[SpacingKey//2::SpacingKey, SpacingKey//2::SpacingKey],
        xmesh[SpacingKey//2::SpacingKey, SpacingKey//2::SpacingKey],
        vy[SpacingKey//2::SpacingKey,    SpacingKey//2::SpacingKey],
        vx[SpacingKey//2::SpacingKey,    SpacingKey//2::SpacingKey],
        scale=ScaleKey,
        headlength=0, headaxislength=0,
        pivot='middle',
        color='yellow',
        angles='xy'
    )

    axes[2].set_title('Fibre Orientation (quiver)', fontsize=13)
    axes[2].set_xticks([])
    axes[2].set_yticks([])

    plt.suptitle(os.path.basename(uploaded_file_path), y=1.02, fontsize=11)
    plt.tight_layout()

    out_path = os.path.join(
        os.path.dirname(uploaded_file_path),
        'Orientation_' + os.path.splitext(os.path.basename(uploaded_file_path))[0] + '.png'
    )
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()